In [1]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

CUDA: True
GPU: Tesla T4


In [2]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

Upload thali_dataset.npz ...


Saving thali_dataset.npz to thali_dataset.npz
Uploaded: ['thali_dataset.npz']


In [3]:
!pip install imageio[ffmpeg] tinycudann -q


ERROR: Could not find a version that satisfies the requirement tinycudann (from versions: none)
ERROR: No matching distribution found for tinycudann


In [4]:
import math, time, json
from pathlib import Path
import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import imageio.v3 as iio

OUT_DIR = Path('/content/nerf_output')
OUT_DIR.mkdir(exist_ok=True)

N_ITERS    = 20_000  
LR         = 1e-3     
BATCH_RAYS = 4096     
N_COARSE   = 96
N_FINE     = 192
LOG_EVERY  = 500
CHUNK      = 1024 * 64

try:
    import tinycudann as tcnn
    HAS_TCNN = True
    print('tinycudann available ')
except ImportError:
    HAS_TCNN = False
    print('falling back to PE')

class HashNeRF(nn.Module):
    def __init__(self, use_hash=True):
        super().__init__()
        self.use_hash = use_hash and HAS_TCNN

        if self.use_hash:
            self.pos_enc = tcnn.Encoding(3, {
                'otype': 'HashGrid',
                'n_levels': 16,
                'n_features_per_level': 2,
                'log2_hashmap_size': 19,
                'base_resolution': 16,
                'per_level_scale': 1.4472,
            })
            pos_dim = 32  # 16 levels * 2 features

            self.dir_enc = tcnn.Encoding(3, {
                'otype': 'SphericalHarmonics',
                'degree': 4,
            })
            dir_dim = 16  # SH degree 4 = 16 coeffs

            self.density_mlp = tcnn.Network(pos_dim, 16, {
                'otype': 'FullyFusedMLP',
                'activation': 'ReLU',
                'output_activation': 'None',
                'n_neurons': 64,
                'n_hidden_layers': 1,
            })
            self.color_mlp = tcnn.Network(16 + dir_dim, 3, {
                'otype': 'FullyFusedMLP',
                'activation': 'ReLU',
                'output_activation': 'Sigmoid',
                'n_neurons': 64,
                'n_hidden_layers': 2,
            })
        else:
            # Fallback standard PE and MLP 
            L_POS = 10; L_DIR = 4; H = 256
            self.freqs_pos = 2.0 ** torch.arange(L_POS)
            self.freqs_dir = 2.0 ** torch.arange(L_DIR)
            pos_dim = 3 + 6*L_POS; dir_dim = 3 + 6*L_DIR
            self.density_mlp = nn.Sequential(
                nn.Linear(pos_dim, H), nn.ReLU(),
                nn.Linear(H, H), nn.ReLU(),
                nn.Linear(H, H), nn.ReLU(),
                nn.Linear(H, H), nn.ReLU(),
                nn.Linear(H, 16),
            )
            self.color_mlp = nn.Sequential(
                nn.Linear(16+dir_dim, H//2), nn.ReLU(),
                nn.Linear(H//2, 3), nn.Sigmoid(),
            )

    def _pe(self, x, freqs):
        out = [x]
        for f in freqs.to(x.device):
            out += [torch.sin(f*math.pi*x), torch.cos(f*math.pi*x)]
        return torch.cat(out, dim=-1)

    def forward(self, pos, dirs):
        if self.use_hash:
            pos01 = (pos + 2.0) / 4.0  
            pos01 = pos01.clamp(0, 1)
            h     = self.pos_enc(pos01)
            feat  = self.density_mlp(h)
            sigma = torch.relu(feat[..., 0])
            dirs01 = (dirs + 1.0) / 2.0
            dh     = self.dir_enc(dirs01)
            rgb    = self.color_mlp(torch.cat([feat, dh], dim=-1))
            rgb    = rgb.float()
            sigma  = sigma.float()
        else:
            h     = self._pe(pos, self.freqs_pos)
            feat  = self.density_mlp(h)
            sigma = torch.relu(feat[..., 0])
            dh    = self._pe(dirs, self.freqs_dir)
            rgb   = self.color_mlp(torch.cat([feat, dh], dim=-1))
        return rgb, sigma

def get_rays(H, W, focal, c2w):
    device = c2w.device
    i, j = torch.meshgrid(
        torch.arange(W, dtype=torch.float32, device=device),
        torch.arange(H, dtype=torch.float32, device=device), indexing='xy')
    dirs = torch.stack([(i-W*0.5)/focal, -(j-H*0.5)/focal, -torch.ones_like(i)], dim=-1)
    R = c2w[:3,:3]
    rays_d = (dirs[...,None,:]*R).sum(-1)
    rays_d = rays_d / torch.norm(rays_d, dim=-1, keepdim=True)
    rays_o = c2w[:3,3].expand_as(rays_d)
    return rays_o.reshape(-1,3), rays_d.reshape(-1,3)

def sample_coarse(rays_o, rays_d, near, far, N):
    n = rays_o.shape[0]
    t = torch.linspace(near, far, N, device=rays_o.device)
    t = t[None].expand(n,-1) + torch.rand(n,N,device=rays_o.device)*(far-near)/N
    return rays_o[:,None,:]+rays_d[:,None,:]*t[:,:,None], t

def sample_fine(rays_o, rays_d, t_c, weights, N_fine):
    weights = weights.float() + 1e-5
    pdf = weights / weights.sum(-1, keepdim=True)
    cdf = torch.cumsum(pdf, -1)
    cdf = torch.cat([torch.zeros_like(cdf[...,:1]), cdf], -1)
    u   = torch.rand(*cdf.shape[:-1], N_fine, device=rays_o.device)
    inds  = torch.searchsorted(cdf.detach(), u, right=True)
    below = (inds-1).clamp(min=0); above = inds.clamp(max=cdf.shape[-1]-1)
    ig = torch.stack([below,above],-1)
    cdf_g  = torch.gather(cdf, 1, ig.reshape(*ig.shape[:-2],-1)).reshape(*ig.shape)
    bins_g = torch.gather(t_c, 1, ig.reshape(*ig.shape[:-2],-1)).reshape(*ig.shape)
    denom  = (cdf_g[...,1]-cdf_g[...,0]).clamp(min=1e-5)
    t_f    = bins_g[...,0] + (u-cdf_g[...,0])/denom*(bins_g[...,1]-bins_g[...,0])
    t_all, _ = torch.sort(torch.cat([t_c, t_f], -1), -1)
    return rays_o[:,None,:]+rays_d[:,None,:]*t_all[:,:,None], t_all

def volume_render(rgb, sigma, t, white_bg=True):
    rgb = rgb.float(); sigma = sigma.float()
    deltas = t[:,1:]-t[:,:-1]
    deltas = torch.cat([deltas, torch.full((deltas.shape[0],1),1e10,device=t.device)],-1)
    alpha  = 1.0 - torch.exp(-sigma*deltas)
    T = torch.cumprod(torch.cat([torch.ones(*alpha.shape[:1],1,device=alpha.device),
                                  1.0-alpha[:,:-1]+1e-10],-1),-1)
    w = T*alpha
    out = (w.unsqueeze(-1)*rgb).sum(1)
    if white_bg: out = out + (1.0 - w.sum(-1, keepdim=True))
    return out, w

def render_rays(model_c, model_f, rays_o, rays_d, near, far, N_c, N_f, chunk=CHUNK):
    N = rays_o.shape[0]
    f_all=[]; c_all=[]
    for i in range(0, N, chunk):
        ro=rays_o[i:i+chunk]; rd=rays_d[i:i+chunk]; n=ro.shape[0]
        pts_c, t_c   = sample_coarse(ro, rd, near, far, N_c)
        dc = rd[:,None,:].expand_as(pts_c)
        rgb_c, sig_c = model_c(pts_c.reshape(-1,3), dc.reshape(-1,3))
        rgb_c=rgb_c.reshape(n,N_c,3); sig_c=sig_c.reshape(n,N_c)
        rc, wc = volume_render(rgb_c, sig_c, t_c)
        with torch.no_grad(): wm=(wc[:,:-1]+wc[:,1:])/2
        pts_f, t_f   = sample_fine(ro, rd, t_c, wm, N_f)
        df = rd[:,None,:].expand_as(pts_f)
        rgb_f, sig_f = model_f(pts_f.reshape(-1,3), df.reshape(-1,3))
        rgb_f=rgb_f.reshape(n,-1,3); sig_f=sig_f.reshape(n,-1)
        rf, _        = volume_render(rgb_f, sig_f, t_f)
        c_all.append(rc); f_all.append(rf)
    return torch.cat(f_all,0), torch.cat(c_all,0)

def psnr(mse): return -10.0*np.log10(mse+1e-10)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

data     = np.load('thali_dataset.npz')
imgs     = torch.from_numpy(data['images_train'].astype(np.float32)/255.0)
c2ws     = torch.from_numpy(data['c2ws_train'].astype(np.float32))
focal    = float(data['focal'])
imgs_val = torch.from_numpy(data['images_val'].astype(np.float32)/255.0)
c2ws_val = torch.from_numpy(data['c2ws_val'].astype(np.float32))
N,H,W,_ = imgs.shape
print(f'train: {N} images | {W}x{H} | focal={focal:.1f}px')

# normalize scene
all_t  = torch.cat([c2ws[:,:3,3], c2ws_val[:,:3,3]], 0)
center = all_t.mean(0)
c2ws[:,:3,3]     -= center
c2ws_val[:,:3,3] -= center
scale  = torch.norm(all_t-center, dim=-1).max().item() * 1.1
c2ws[:,:3,3]     /= scale
c2ws_val[:,:3,3] /= scale
cam_d  = torch.norm(c2ws[:,:3,3], dim=-1)
NEAR   = max(0.01, float(cam_d.min())-0.5)
FAR    = float(cam_d.max()) + 1.5
print(f'scale={scale:.4f}m | NEAR={NEAR:.3f} | FAR={FAR:.3f}')

print('computing rays...')
all_ro=[]; all_rd=[]; all_rgb=[]
for i in range(N):
    ro,rd=get_rays(H,W,focal,c2ws[i])
    all_ro.append(ro); all_rd.append(rd); all_rgb.append(imgs[i].reshape(-1,3))
all_ro=torch.cat(all_ro,0); all_rd=torch.cat(all_rd,0); all_rgb=torch.cat(all_rgb,0)
print(f'rays: {len(all_ro):,}')

model_c = HashNeRF(use_hash=HAS_TCNN).to(device)
model_f = HashNeRF(use_hash=HAS_TCNN).to(device)
params  = list(model_c.parameters())+list(model_f.parameters())
optim   = torch.optim.Adam(params, lr=LR, eps=1e-15)
sched   = torch.optim.lr_scheduler.CosineAnnealingLR(optim, N_ITERS, eta_min=LR*0.01)
scaler  = torch.amp.GradScaler('cuda')
loss_fn = nn.MSELoss()
psnr_log=[]; t0=time.time()

for step in tqdm(range(1, N_ITERS+1)):
    idx    = torch.randint(0, len(all_ro), (BATCH_RAYS,))
    ro     = all_ro[idx].to(device)
    rd     = all_rd[idx].to(device)
    target = all_rgb[idx].to(device)
    with torch.amp.autocast('cuda'):
        rgb_f, rgb_c = render_rays(model_c, model_f, ro, rd, NEAR, FAR, N_COARSE, N_FINE)
        loss = loss_fn(rgb_f, target) + 0.5*loss_fn(rgb_c, target)
    optim.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optim)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(optim); scaler.update(); sched.step()
    if step%LOG_EVERY==0 or step==1:
        with torch.no_grad():
            p=psnr(float(loss_fn(rgb_f.detach().float(), target).item()))
        psnr_log.append((step,p))
        print(f'  step {step:6d} | loss {loss.detach().item():.5f} | psnr {p:.2f} dB | {time.time()-t0:.0f}s')

print('Training done')

tinycudann not available - falling back to PE
Device: cuda
train: 40 images | 400x300 | focal=333.6px
scale=0.4703m | NEAR=0.010 | FAR=2.409
computing rays...
rays: 4,800,000


  0%|          | 0/20000 [00:00<?, ?it/s]

  step      1 | loss 0.19874 | psnr 12.07 dB | 2s
  step    500 | loss 0.16250 | psnr 15.59 dB | 177s
  step   1000 | loss 0.15648 | psnr 16.98 dB | 355s
  step   1500 | loss 0.15396 | psnr 17.49 dB | 532s
  step   2000 | loss 0.14725 | psnr 18.07 dB | 709s
  step   2500 | loss 0.15157 | psnr 18.51 dB | 886s
  step   3000 | loss 0.15242 | psnr 18.60 dB | 1063s
  step   3500 | loss 0.14756 | psnr 18.93 dB | 1240s
  step   4000 | loss 0.15032 | psnr 18.86 dB | 1417s
  step   4500 | loss 0.14972 | psnr 19.10 dB | 1593s
  step   5000 | loss 0.14352 | psnr 19.47 dB | 1770s
  step   5500 | loss 0.14546 | psnr 19.69 dB | 1946s
  step   6000 | loss 0.14787 | psnr 20.08 dB | 2123s
  step   6500 | loss 0.14426 | psnr 19.81 dB | 2299s
  step   7000 | loss 0.14610 | psnr 19.59 dB | 2476s
  step   7500 | loss 0.14368 | psnr 20.35 dB | 2653s
  step   8000 | loss 0.14152 | psnr 20.43 dB | 2829s
  step   8500 | loss 0.14536 | psnr 20.53 dB | 3006s
  step   9000 | loss 0.14252 | psnr 20.40 dB | 3182s
 

In [8]:
import gc
torch.cuda.empty_cache()
gc.collect()

RENDER_CHUNK = 1024  # much smaller than training chunk

print('Rendering val image')
model_c.eval(); model_f.eval()
with torch.no_grad():
    ro, rd = get_rays(H, W, focal, c2ws_val[0].to(device))
    rgb_out, _ = render_rays(model_c, model_f, ro, rd, NEAR, FAR,
                             N_COARSE, N_FINE, chunk=RENDER_CHUNK)
rendered = (rgb_out.cpu().float().numpy().reshape(H,W,3)*255).clip(0,255).astype(np.uint8)
gt = (imgs_val[0].numpy()*255).astype(np.uint8)

fig, axes = plt.subplots(1,2,figsize=(10,5))
axes[0].imshow(gt);       axes[0].set_title('Ground Truth'); axes[0].axis('off')
axes[1].imshow(rendered); axes[1].set_title('NeRF v3 Render 22dB'); axes[1].axis('off')
fig.tight_layout(); fig.savefig(OUT_DIR/'val_render.png', dpi=150); plt.show()
iio.imwrite(OUT_DIR/'val_render_rgb.png', rendered)

print('Rendering spiral (60 frames)')
frames = []; c2w0 = c2ws_val[0].numpy()
for fi in tqdm(range(60)):
    a = fi/60*2*math.pi
    rot = np.array([[math.cos(a),0,math.sin(a),0],
                    [0,1,0,0],
                    [-math.sin(a),0,math.cos(a),0],
                    [0,0,0,1]], dtype=np.float32)
    c2wi = torch.from_numpy(rot @ c2w0).to(device)
    with torch.no_grad():
        ro, rd = get_rays(H, W, focal, c2wi)
        ri, _ = render_rays(model_c, model_f, ro, rd, NEAR, FAR,
                            N_COARSE, N_FINE, chunk=RENDER_CHUNK)
    frames.append((ri.cpu().float().numpy().reshape(H,W,3)*255).clip(0,255).astype(np.uint8))

iio.imwrite(OUT_DIR/'spiral.gif', frames, duration=1/24, loop=0)

sl, ps = zip(*psnr_log)
plt.figure(figsize=(8,4)); plt.plot(sl, ps, marker='o')
plt.xlabel('Step'); plt.ylabel('PSNR (dB)'); plt.title('NeRF v3 PSNR'); plt.grid(True)
plt.tight_layout(); plt.savefig(OUT_DIR/'psnr_curve.png', dpi=120); plt.show()

torch.save({'c': model_c.state_dict(), 'f': model_f.state_dict(),
            'center': center, 'scale': scale, 'NEAR': NEAR, 'FAR': FAR},
           OUT_DIR/'nerf_weights.pt')
print('saved')

# Download
import shutil
from google.colab import files
shutil.make_archive('/content/nerf_results', 'zip', '/content/nerf_output')
files.download('/content/nerf_results.zip')

Rendering val image...
Rendering spiral (60 frames)...


  0%|          | 0/60 [00:00<?, ?it/s]

saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import shutil
from google.colab import files
shutil.make_archive('/content/nerf_results','zip','/content/nerf_output')
files.download('/content/nerf_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>